# Practical 2 — Email Spam Classification using KNN and SVM

**Aim:** Classify an email as **Spam (1)** or **Not Spam (0)** using K-Nearest Neighbors (KNN) and Support Vector Machine (SVM), then compare their performance.

**Dataset:** `emails.csv` (keep it in the same folder as this notebook).

**Learning order:** Load → Clean → X/y → Split → Scale → KNN → SVM → Compare.

> Each section has a short explanation followed by runnable code. Run cells **from top to bottom** using Shift + Enter.

## 1. Import libraries

A **library** is a toolbox of ready-made Python functions.
- **pandas:** opens and manages CSV data.
- **train_test_split:** divides data into training and testing sets.
- **StandardScaler:** puts numeric features on comparable scales.
- **KNeighborsClassifier:** KNN classification algorithm.
- **SVC:** Support Vector Machine classifier.
- **accuracy_score, confusion_matrix, precision_score, recall_score:** evaluate predictions.

In [ ]:
# Import only the libraries we need
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score

## 2. Load and understand the dataset

Each **row** is an email. The word columns (like `the`, `to`, `you`) contain numeric word counts. `Email No.` is an identifier, not a useful input. `Prediction` is the correct answer: **0 = not spam**, **1 = spam**.

The supplied dataset has **5,172 rows** and **3,002 columns** before cleaning.

In [ ]:
# Read the CSV file from the current working folder
df = pd.read_csv("emails.csv")

print("Dataset shape:", df.shape)  # (rows, columns)
print(df.head())                    # Show first 5 emails
print("Missing values:", df.isnull().sum().sum())

## 3. Clean the data and choose X and y

- `dropna()` removes rows with missing data.
- `X` contains the **input** word counts (questions).
- `y` contains the **output** spam label (answers).
- We exclude `Email No.` because it is only an ID.

In [ ]:
# Remove rows containing missing values
df = df.dropna()

# X = all word-count columns; remove ID and answer column
X = df.drop(columns=["Email No.", "Prediction"])

# y = the correct spam label (0 or 1)
y = df["Prediction"]

print("Inputs:", X.shape)
print("Outputs:", y.shape)

## 4. Split into training and testing data

**Training (80%)** teaches the model. **Testing (20%)** checks how well it predicts emails it has not seen.

`random_state=42` makes the random split repeatable. `stratify=y` keeps approximately the same spam/not-spam proportion in both sets.

In [ ]:
# Divide the dataset: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## 5. Scale the input features

KNN measures **distance**, so large numeric values can dominate without scaling. Scaling is also useful for SVM.

- `fit_transform(X_train)` learns mean/standard deviation from **training data**, then scales it.
- `transform(X_test)` applies the **same scaling** to test data.

**Memory trick:** Train = fit + transform; Test = transform only.

In [ ]:
# Standardize word-count features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## 6. KNN — K-Nearest Neighbors

KNN predicts an email's label by checking its **K closest training examples**. If most nearby emails are spam, it predicts spam.

`n_neighbors=5` means look at the 5 nearest emails. `fit()` stores/learns from training examples; `predict()` makes predictions for test emails.

In [ ]:
# Create and train KNN model
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

# Predict spam (1) or not spam (0)
knn_pred = knn.predict(X_test)
print("KNN accuracy:", accuracy_score(y_test, knn_pred))

## 7. SVM — Support Vector Machine

SVM learns a **decision boundary** that separates spam from non-spam emails.

`kernel="linear"` uses a straight separating boundary in the feature space. This is simpler and usually faster for datasets with many word features than an RBF kernel. Your original notebook uses `SVC(C=1)` with its default RBF kernel; changing the kernel may change the scores.

In [ ]:
# Create and train SVM model
svm = SVC(kernel="linear", C=1)
svm.fit(X_train, y_train)

# Predict spam (1) or not spam (0)
svm_pred = svm.predict(X_test)
print("SVM accuracy:", accuracy_score(y_test, svm_pred))

## 8. Compare model performance

**Accuracy:** fraction of emails classified correctly (higher is better).

**Confusion matrix:** `[[TN, FP], [FN, TP]]` where:
- **TN:** non-spam correctly predicted non-spam
- **FP:** non-spam wrongly predicted spam
- **FN:** spam wrongly predicted non-spam
- **TP:** spam correctly predicted spam

**Precision:** Of emails predicted spam, how many really were spam?

**Recall:** Of actual spam emails, how many did the model catch?

**Error rate = 1 − accuracy**. `zero_division=0` avoids warnings when a score cannot be calculated.

In [ ]:
# Print the same evaluation measures for both models
for name, pred in [("KNN", knn_pred), ("SVM", svm_pred)]:
    accuracy = accuracy_score(y_test, pred)
    print("\n", name)
    print("Confusion Matrix:\n", confusion_matrix(y_test, pred))
    print("Accuracy:", accuracy)
    print("Error Rate:", 1 - accuracy)
    print("Precision:", precision_score(y_test, pred, zero_division=0))
    print("Recall:", recall_score(y_test, pred, zero_division=0))

## 9. How to interpret the output

Compare the **test accuracy** of KNN and SVM; higher is better for overall correctness. Also look at **precision** and **recall**: if missing spam is costly, recall matters; if wrongly blocking genuine mail is costly, precision matters.

**Do not memorize made-up scores.** Run the notebook and read the numbers produced on your own computer.

## 10. Quick viva questions (simple answers)

1. **What is classification?** Predicting a category, such as spam or not spam.
2. **What is supervised learning?** Training with examples that have known correct answers.
3. **What is KNN?** It classifies using the majority label of nearby examples.
4. **What does K=5 mean?** Check the five nearest examples.
5. **What is SVM?** A classifier that learns a boundary separating classes.
6. **What is a feature?** An input column; here, a word-count column.
7. **What is the target?** `Prediction`, where 0 means not spam and 1 means spam.
8. **Why remove `Email No.`?** It is an identifier, not meaningful email content.
9. **Why split data?** To test the model on unseen emails.
10. **Why scale?** So feature magnitudes do not unfairly dominate distance calculations.
11. **What does `fit()` do?** Trains the model.
12. **What does `predict()` do?** Predicts labels for test examples.
13. **What is accuracy?** Correct predictions divided by all predictions.
14. **What is a confusion matrix?** A table of correct and incorrect predictions by class.
15. **What is precision?** Correctly predicted spam divided by all predicted spam.
16. **What is recall?** Correctly predicted spam divided by all actual spam.
17. **What is overfitting?** Doing well on training data but poorly on new data.

### Memory trick

**Import → Read → Clean → X/y → Split → Scale → KNN → SVM → Evaluate**

### Common errors
- **FileNotFoundError:** put `emails.csv` in the notebook working folder.
- **ModuleNotFoundError:** run `%pip install pandas scikit-learn` in a separate notebook cell.
- **No kernel:** install the Microsoft Python and Jupyter extensions in VS Code, then select your Python environment.

**Exam tip:** Keep this notebook and `emails.csv` together. Run cells in order.